<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.2/3_memory_persistent_mem0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 3. Memoria Persistente y Semántica con LangChain + Mem0

## Objetivos de Aprendizaje

Al finalizar este notebook podrás:

- Diferenciar la **memoria conversacional** de una **memoria persistente y semántica**.
- Comprender que un LLM no "recuerda" por sí solo: necesita un mecanismo externo para almacenar y recuperar información.
- Guardar recuerdos relevantes utilizando **Mem0 OSS**.
- Buscar recuerdos utilizando **similitud semántica**.
- Incorporar recuerdos recuperados al contexto de un LLM con LangChain.
- Comparar una respuesta **sin memoria** con otra **enriquecida con memoria**.
- Comprender por qué la memoria debe mantenerse separada entre distintos usuarios.

---

## ¿Qué cambia respecto de los notebooks anteriores?

Hasta ahora trabajamos con memoria conversacional:

```text
Buffer Memory
→ conserva toda la conversación

Window Memory
→ conserva solo los últimos turnos

Summary Memory
→ conserva un resumen de la conversación
```

En esos casos, la memoria dependía principalmente del historial de conversación.

Ahora veremos otro enfoque:

```text
Memoria persistente / semántica
        ↓
extrae información relevante
        ↓
la almacena fuera del chat_history
        ↓
la recupera cuando una nueva pregunta la hace útil
```

> **Idea clave:** no necesitamos reenviar toda la conversación anterior para recordar una preferencia del usuario.


### 1. Instalación y Configuración

In [14]:

# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys

if "google.colab" in sys.modules:
    !pip install -q mem0ai qdrant-client langchain langchain-openai python-dotenv


In [2]:

# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os

try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets

    # Solo LLM_API_KEY es obligatorio.
    # Los demás valores tienen defaults para mantener el mismo esquema
    # utilizado en los notebooks anteriores.
    for _k in (
        "LLM_API_KEY",
        "LANGSMITH_API_KEY",
        "LLM_BASE_URL",
        "LLM_MODEL",
        "LLM_MODEL_SMALL",
        "LLM_EMBEDDING_MODEL",
    ):
        try:
            value = userdata.get(_k)
            if value:
                os.environ[_k] = value
        except Exception:
            pass                               # el Secret no existe: se usa el default

    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
    os.environ.setdefault("LLM_EMBEDDING_MODEL", "mistral-embed")

except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

print("🔍 Configuración disponible:")
print(f"LLM_BASE_URL        : {os.environ.get('LLM_BASE_URL', '❌ No configurado')}")
print(f"LLM_MODEL           : {os.environ.get('LLM_MODEL', '❌ No configurado')}")
print(f"LLM_EMBEDDING_MODEL : {os.environ.get('LLM_EMBEDDING_MODEL', '❌ No configurado')}")
print(f"LLM_API_KEY         : {'✅ Configurado' if os.environ.get('LLM_API_KEY') else '❌ No configurado'}")


🔍 Configuración disponible:
LLM_BASE_URL        : https://api.mistral.ai/v1
LLM_MODEL           : ministral-14b-latest
LLM_EMBEDDING_MODEL : mistral-embed
LLM_API_KEY         : ✅ Configurado



### 2. Configuración del LLM de LangChain

Seguimos utilizando el mismo patrón de los notebooks anteriores.

Este LLM será el encargado de **generar las respuestas al usuario**.

Más adelante Mem0 utilizará también un modelo de lenguaje y un modelo de embeddings para organizar y recuperar recuerdos.


In [3]:

from langchain_openai import ChatOpenAI

try:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL", "mistral-small-latest"),
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=0
    )

    test = llm.invoke("Responde solamente: funcionando")
    print("✅ LLM de LangChain configurado.")
    print(f"📝 Respuesta de prueba: {test.content}")

except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None


✅ LLM de LangChain configurado.
📝 Respuesta de prueba: ¡Funcionando! 🚀



### 3. Configuración de Mem0 OSS

Mem0 necesita tres piezas principales:

1. **LLM**: ayuda a identificar qué información vale la pena recordar.
2. **Embeddings**: representan los recuerdos numéricamente para poder buscarlos por similitud semántica.
3. **Vector Store**: almacena esos recuerdos y sus representaciones.

En este ejemplo usamos:

- el mismo endpoint y API Key de los notebooks anteriores;
- `mistral-small-latest` como LLM;
- `mistral-embed` para embeddings;
- **Qdrant local** como almacenamiento vectorial.

> La base de Qdrant se guarda en el disco temporal de Colab. Por eso la memoria persiste entre celdas y conversaciones dentro del mismo runtime, pero se pierde si reiniciamos completamente el entorno de Colab.


In [4]:

from mem0 import Memory

# Mem0 permite configurar explícitamente cada componente.
# Usamos el proveedor "openai" porque Mistral expone una API compatible
# con el formato OpenAI para chat y embeddings.

mem0_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": os.getenv("LLM_MODEL", "mistral-small-latest"),
            "api_key": os.getenv("LLM_API_KEY"),
            "openai_base_url": os.getenv("LLM_BASE_URL"),
            "temperature": 0,
            "top_p": 1,
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": os.getenv("LLM_EMBEDDING_MODEL", "mistral-embed"),
            "api_key": os.getenv("LLM_API_KEY"),
            "openai_base_url": os.getenv("LLM_BASE_URL"),
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "path": "/tmp/mem0_qdrant",
            "collection_name": "memoria_clase",
            "embedding_model_dims": 1024,
        },
    },
}

try:
    memory = Memory.from_config(mem0_config)
    print("✅ Mem0 configurado correctamente.")
except Exception as e:
    print(f"❌ Error configurando Mem0: {e}")
    memory = None


✅ Mem0 configurado correctamente.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



### 4. Guardando un primer recuerdo

Usaremos un ejemplo muy simple de cultura general.

El usuario comenta:

> "Mi planeta favorito es Saturno y me interesan especialmente sus anillos."

Mem0 puede analizar esa interacción y almacenar información relevante asociada a un identificador de usuario.

El `user_id` es muy importante: permite separar la memoria de distintas personas.


In [5]:

USER_ID = "estudiante_demo"

messages = [
    {
        "role": "user",
        "content": "Mi planeta favorito es Saturno y me interesan especialmente sus anillos."
    },
    {
        "role": "assistant",
        "content": "Entendido. Tendré en cuenta tu interés por Saturno y sus anillos."
    }
]

if memory is not None:
    result_add = memory.add(
        messages,
        user_id=USER_ID
    )

    print("✅ Interacción enviada a Mem0.")
    print("📝 Resultado de la operación:")
    print(result_add)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/mem0/vector_stores/qdrant.py:177: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(


✅ Interacción enviada a Mem0.
📝 Resultado de la operación:
{'results': [{'id': 'e08f557c-ac6f-400c-a76b-d1965118fd12', 'memory': "User's favorite planet is Saturn and they have a particular interest in its rings", 'event': 'ADD'}]}



### 5. ¿Qué quedó realmente almacenado?

Una memoria semántica no tiene por qué conservar literalmente toda la conversación.

Mem0 puede extraer un recuerdo más compacto, por ejemplo:

```text
Al usuario le interesa Saturno y especialmente sus anillos.
```

Inspeccionemos la memoria asociada al usuario.


In [6]:

if memory is not None:
    all_memories = memory.get_all(
        filters={"user_id": USER_ID}
    )

    print("🧠 Recuerdos almacenados:")
    print()

    for item in all_memories.get("results", []):
        print(f"- {item.get('memory')}")


🧠 Recuerdos almacenados:

- User's favorite planet is Saturn and they have a particular interest in its rings



### 6. Primera comparación: responder SIN usar la memoria

Ahora iniciaremos una nueva interacción.

La pregunta **no menciona Saturno**:

> "Recomiéndame otro planeta que podría interesarme y explícame por qué."

Primero enviaremos la pregunta directamente al LLM.

El modelo puede entregar una respuesta razonable, pero no conoce la preferencia que expresamos anteriormente porque no le estamos enviando el historial ni los recuerdos.


In [7]:

question = "Recomiéndame otro planeta que podría interesarme y explícame por qué."

if llm is not None:
    response_without_memory = llm.invoke(question)

    print("🤖 RESPUESTA SIN MEMORIA")
    print("-" * 60)
    print(response_without_memory.content)


🤖 RESPUESTA SIN MEMORIA
------------------------------------------------------------
¡Excelente pregunta! Si te interesa explorar mundos fascinantes más allá de la Tierra, te recomiendo **Europa**, una de las lunas de Júpiter. Aquí te explico por qué es tan intrigante:

### **1. Océano global bajo su corteza de hielo**
Europa es uno de los lugares más prometedores en la búsqueda de **vida extraterrestre** en nuestro sistema solar. Bajo su superficie helada (de unos 15–25 km de grosor) se esconde un **océano global de agua líquida**, mantenido cálido por las fuerzas de marea de Júpiter. Este océano podría contener **más agua que todos los océanos de la Tierra juntos** y, según modelos científicos, podría tener condiciones similares a las de los lagos subglaciales de la Antártida en la Tierra, donde ya se ha encontrado vida microbiana.

### **2. Actividad geológica y posible hidrotérmica**
Imágenes de la sonda **Galileo** (NASA) revelaron una superficie joven y llena de grietas, lo que s

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



### 7. Recuperación semántica de recuerdos

Ahora preguntaremos a Mem0 qué recuerdos podrían ser relevantes para responder la misma pregunta.

La búsqueda es **semántica**: no exige que la nueva pregunta contenga exactamente las mismas palabras que el recuerdo.

```text
Pregunta nueva
      ↓
embedding
      ↓
comparación con recuerdos almacenados
      ↓
recuerdos más relacionados
```

Siempre restringimos la búsqueda por `user_id` para evitar mezclar información de distintos usuarios.


In [8]:

if memory is not None:
    search_result = memory.search(
        question,
        filters={"user_id": USER_ID},
        top_k=5
    )

    relevant_memories = search_result.get("results", [])

    print("🔎 Recuerdos recuperados:")
    print()

    for item in relevant_memories:
        print(f"- {item.get('memory')}")


🔎 Recuerdos recuperados:

- User's favorite planet is Saturn and they have a particular interest in its rings



### 8. Incorporando los recuerdos al prompt con LangChain

Mem0 **recupera la información**, pero el LLM necesita recibirla dentro de su contexto para poder utilizarla.

Construiremos un prompt muy simple:

```text
Recuerdos relevantes
        +
Pregunta actual
        ↓
LLM
        ↓
Respuesta personalizada
```

Este patrón permite mantener separadas dos responsabilidades:

- **Mem0** decide qué recuerdos recuperar.
- **LangChain + LLM** utilizan esos recuerdos para generar la respuesta.


In [9]:

from langchain_core.prompts import ChatPromptTemplate

# Convertimos los recuerdos recuperados en texto legible para el LLM.
memory_text = "\n".join(
    f"- {item.get('memory')}"
    for item in relevant_memories
)

if not memory_text:
    memory_text = "- No se encontraron recuerdos relevantes."

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Eres un asistente útil.

Puedes utilizar recuerdos relevantes del usuario para personalizar tu respuesta.
No inventes recuerdos que no aparezcan en el contexto entregado.

Recuerdos relevantes:
{memories}
"""
    ),
    ("human", "{question}")
])

chain_with_memory = prompt | llm

response_with_memory = chain_with_memory.invoke({
    "memories": memory_text,
    "question": question
})

print("🧠 RESPUESTA CON MEMORIA")
print("-" * 60)
print(response_with_memory.content)


🧠 RESPUESTA CON MEMORIA
------------------------------------------------------------
¡Excelente elección con Saturno! Dado tu interés en sus **anillos icónicos**, te recomendaría explorar **Júpiter**, aunque con un enfoque diferente pero igualmente fascinante.

### **¿Por qué Júpiter podría interesarte?**
1. **Sistema de anillos ocultos (pero espectaculares)**:
   Aunque no son tan visibles como los de Saturno, Júpiter tiene **anillos tenues y complejos**, compuestos principalmente por polvo. Fueron descubiertos por la sonda *Voyager 1* en 1979 y luego estudiados en detalle por *Galileo* y *Juno*. Son más difíciles de observar desde la Tierra, pero su formación y dinámica son un misterio científico apasionante.

2. **Lunas fascinantes (¡como un mini sistema solar!)**:
   Júpiter tiene **79 lunas confirmadas**, pero cuatro destacan por su singularidad:
   - **Ío**: El cuerpo más volcánico del sistema solar, con géiseres de azufre y lava en constante erupción.
   - **Europa**: Tiene un o

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



### 9. Comparando ambos comportamientos

El modelo es el mismo en los dos casos.

Lo que cambia es el **contexto disponible**:

```text
SIN MEMORIA
Pregunta → LLM → respuesta genérica

CON MEMORIA
Pregunta
   +
recuerdos relevantes
   ↓
LLM
   ↓
respuesta contextualizada
```

Esta distinción es fundamental:

> **La memoria no cambia el conocimiento interno del modelo. Cambia la información que ponemos a su disposición cuando responde.**


In [10]:

print("📌 COMPARACIÓN")
print("=" * 70)

print("\nSIN MEMORIA:")
print(response_without_memory.content)

print("\n" + "-" * 70)

print("\nCON MEMORIA:")
print(response_with_memory.content)


📌 COMPARACIÓN

SIN MEMORIA:
¡Excelente pregunta! Si te interesa explorar mundos fascinantes más allá de la Tierra, te recomiendo **Europa**, una de las lunas de Júpiter. Aquí te explico por qué es tan intrigante:

### **1. Océano global bajo su corteza de hielo**
Europa es uno de los lugares más prometedores en la búsqueda de **vida extraterrestre** en nuestro sistema solar. Bajo su superficie helada (de unos 15–25 km de grosor) se esconde un **océano global de agua líquida**, mantenido cálido por las fuerzas de marea de Júpiter. Este océano podría contener **más agua que todos los océanos de la Tierra juntos** y, según modelos científicos, podría tener condiciones similares a las de los lagos subglaciales de la Antártida en la Tierra, donde ya se ha encontrado vida microbiana.

### **2. Actividad geológica y posible hidrotérmica**
Imágenes de la sonda **Galileo** (NASA) revelaron una superficie joven y llena de grietas, lo que sugiere que el hielo se mueve y se renueva constantemente.


### 10. Agregando una nueva preferencia

La memoria puede crecer a medida que aparecen nuevos antecedentes.

Ahora el usuario agrega:

> "También me interesan mucho las lunas heladas."

Guardaremos esa nueva interacción y después haremos una pregunta distinta.

No necesitamos reenviar la conversación anterior.


In [11]:

new_messages = [
    {
        "role": "user",
        "content": "También me interesan mucho las lunas heladas."
    },
    {
        "role": "assistant",
        "content": "Entendido. Tendré en cuenta también tu interés por las lunas heladas."
    }
]

if memory is not None:
    memory.add(
        new_messages,
        user_id=USER_ID
    )

    print("✅ Nueva información agregada a la memoria.")


✅ Nueva información agregada a la memoria.


In [12]:

new_question = "¿Qué tema del sistema solar me recomendarías estudiar ahora?"

# 1. Recuperamos recuerdos relacionados con la nueva pregunta.
new_search = memory.search(
    new_question,
    filters={"user_id": USER_ID},
    top_k=5
)

new_memories = new_search.get("results", [])

print("🔎 Recuerdos recuperados para la nueva pregunta:")
for item in new_memories:
    print(f"- {item.get('memory')}")

# 2. Los incorporamos al contexto del mismo chain.
new_memory_text = "\n".join(
    f"- {item.get('memory')}"
    for item in new_memories
)

# 3. Generamos una nueva respuesta contextualizada.
new_response = chain_with_memory.invoke({
    "memories": new_memory_text,
    "question": new_question
})

print("\n🧠 RESPUESTA:")
print(new_response.content)


🔎 Recuerdos recuperados para la nueva pregunta:
- User has a strong interest in Saturn's icy moons, in addition to their existing fascination with the planet's rings
- User's favorite planet is Saturn and they have a particular interest in its rings


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



🧠 RESPUESTA:
Dado tu interés previo en **Saturno** y, en particular, en sus **anillos** y sus **lunas heladas** (como Encélado, Titán o Mimas), te recomendaría profundizar en alguno de estos temas fascinantes:

1. **La formación y composición de los anillos de Saturno**:
   - ¿Cómo se originaron? (Hipótesis de colisión de lunas o captura de material).
   - Su estructura detallada: divisiones (como la *División de Cassini*), partículas de hielo y polvo, y su dinámica (ondas espirales, "propellers").
   - Comparación con los anillos de otros gigantes gaseosos (Júpiter, Urano, Neptuno).

2. **Las lunas heladas de Saturno: mundos oceánicos y potencial habitable**:
   - **Encélado**: Sus *geysers de agua* y el océano subterráneo bajo su corteza de hielo (datos de la misión *Cassini*).
   - **Titán**: Su atmósfera densa, lagos de metano/etano y química prebiótica (similar a la Tierra primitiva).
   - **Mimas**: Su cráter *Herschel* y su posible océano interno (sorpresa reciente).
   - **Dio


### 11. Separación de memoria por usuario

En sistemas reales no queremos que los recuerdos de una persona aparezcan en la conversación de otra.

Por eso Mem0 permite asociar los recuerdos a identificadores como `user_id`.

Comprobemos qué ocurre al buscar recuerdos utilizando un usuario diferente.


In [13]:

OTHER_USER_ID = "otro_estudiante"

other_user_memories = memory.search(
    "¿Qué planetas y temas me interesan?",
    filters={"user_id": OTHER_USER_ID},
    top_k=5
)

print("🔐 Recuerdos encontrados para otro usuario:")
print(other_user_memories.get("results", []))


🔐 Recuerdos encontrados para otro usuario:
[]



### 12. ¿Es realmente una memoria "persistente"?

Sí, pero conviene precisar el concepto.

En este notebook los recuerdos ya no dependen de `chat_history`; están almacenados en un **vector store** separado del prompt.

Sin embargo, estamos usando Qdrant en el almacenamiento temporal del runtime de Colab:

```text
Conversación actual
      ≠
Memoria de Mem0
```

pero:

```text
Reinicio completo del runtime de Colab
      ↓
se elimina /tmp
      ↓
se pierde esta base local
```

En una aplicación real, la memoria podría almacenarse en un servicio persistente como Qdrant, PostgreSQL + pgvector u otra base adecuada.

> **Memoria de largo plazo** significa que el conocimiento puede sobrevivir a una conversación particular; la durabilidad física depende del almacenamiento elegido.



## Conclusiones

En este notebook incorporamos una nueva forma de memoria para agentes:

1. **La memoria no está en el LLM.**  
   El modelo necesita que otro componente almacene y recupere información.

2. **Mem0 puede extraer recuerdos relevantes.**  
   No necesitamos conservar literalmente toda la conversación.

3. **La recuperación puede ser semántica.**  
   Una pregunta nueva puede recuperar información relacionada aunque no use exactamente las mismas palabras.

4. **LangChain incorpora los recuerdos al contexto del modelo.**  
   Recuperar memoria y generar una respuesta son responsabilidades distintas.

5. **El `user_id` permite separar recuerdos.**  
   En sistemas multiusuario esta separación es fundamental.

6. **Persistencia no significa necesariamente permanencia.**  
   En este ejemplo los recuerdos sobreviven a distintas interacciones dentro del runtime, pero el almacenamiento local de Colab desaparece al reiniciarlo.

### Preguntas de reflexión

- ¿Qué información vale la pena almacenar como recuerdo?
- ¿Qué información sería riesgoso guardar?
- ¿Qué debería ocurrir si una preferencia del usuario cambia?
- ¿Cómo evitaríamos utilizar recuerdos de una persona en la conversación de otra?



## Otras alternativas para memoria en agentes

Mem0 es una alternativa open source para incorporar memoria persistente y recuperación semántica, pero no es la única.

Algunas alternativas y enfoques relacionados son:

- **LangGraph**: permite manejar estado y memoria dentro de workflows con mayor control sobre el flujo de ejecución.
- **Letta**: está orientado a agentes con estado persistente y memoria de largo plazo integrada.
- **Hindsight**: incorpora memoria basada en experiencias y mecanismos de recuperación y reflexión.
- **Bases vectoriales como Qdrant, Chroma o pgvector**: permiten construir mecanismos de memoria personalizados utilizando embeddings y búsqueda semántica.

### Idea clave

No existe una única forma de implementar memoria.

La elección depende de factores como:

- duración de la memoria;
- persistencia requerida;
- número de usuarios;
- privacidad y seguridad de los datos;
- necesidad de actualizar o eliminar recuerdos;
- complejidad del agente.

En este notebook utilizamos **Mem0** porque permite observar de manera simple el ciclo:

**guardar → recuperar → incorporar al contexto → responder**.
